In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

# Laboratory Exercise 4: A* Algorithm in Python

Members:\
Lamzon, Ellen Faith (2023-60093) \
Malupangue, Carl Cyrill (2023-60041) \
Ringcunada, Ivy (2023-00424)


This program finds the cheapest path from a start point to a goal point
on a small map (graph). Each connection between two points has a cost.\
A* picks which point to visit next using:\
    - f(n) = g(n) + h(n)\
    - g(n) = actual cost from the start to this point\
    - h(n) = estimated (guessed) cost from this point to the goal

In [ ]:
## Define Key Varialbes

initial_state = "S"  # The point where the search starts
goal_state = "G"     # The point we want to reach
 
# state_space is the "map". For each point, it lists the points it
# connects to and how much it costs to move there.
state_space = {
    "S": {"A": 1, "B": 4},           # From S you can go to A (cost 1) or B (cost 4)
    "A": {"B": 2, "C": 5, "D": 12},  # From A you can go to B, C, or D
    "B": {"C": 2},                   # From B you can only go to C
    "C": {"D": 3, "G": 6},           # From C you can go to D or straight to G
    "D": {"G": 1},                   # From D you can go to G
    "G": {}                          # G is the goal, so it has no outgoing moves
}
 
# heuristic holds our guess of the remaining cost from each point to G.
# These guesses never go above the real cost, so they are "admissible".
# That is what guarantees A* finds the cheapest path.
heuristic = {
    "S": 7,  # Guess: about 7 more to reach G from S (real cost is 9)
    "A": 6,  # Guess from A (real cost is 8)
    "B": 4,  # Guess from B (real cost is 6)
    "C": 3,  # Guess from C (real cost is 4)
    "D": 1,  # Guess from D (real cost is 1)
    "G": 0   # Already at the goal, so the guess is 0
}

In [ ]:
## Node Class

class Node:
    """A Node stores one point in the search plus the costs to reach it."""
 
    def __init__(self, state, parent=None, g_cost=0, h_cost=0):
        self.state = state                  # The name of the point (e.g., "A")
        self.parent = parent                # The node we came from (used to rebuild the path)
        self.g_cost = g_cost                # Actual cost from the start to this node
        self.h_cost = h_cost                # Guessed cost from this node to the goal
        self.f_cost = g_cost + h_cost       # Total score A* uses to decide what to visit next
 
    def __repr__(self):
        # Controls how a node looks when printed, e.g. A(g=1, h=6, f=7)
        return f"{self.state}(g={self.g_cost}, h={self.h_cost}, f={self.f_cost})"


In [ ]:
## helper function: rebuild the path

def reconstruct_path(node):
    """Walk backward from the goal to the start using each node's parent."""
    path = []                    # Empty list to collect the points
    while node is not None:      # Keep going until we pass the start (start has no parent)
        path.append(node.state)  # Add the current point to the list
        node = node.parent       # Move one step back
    path.reverse()               # The list is goal-to-start, so flip it to start-to-goal
    return path                  # Give back the finished path

In [ ]:
## The A* Algorithm

def a_star(start, goal, graph, h):
    """Search for the cheapest path from start to goal."""
 
    start_node = Node(start, None, 0, h[start])  # Make the starting node (g = 0, no parent)
 
    open_list = [start_node]  # Open list = nodes found but not yet explored
    closed_list = set()       # Closed list = points already fully explored
    best_g = {start: 0}       # Cheapest known cost to reach each point so far
 
    step = 0  # Counts how many nodes we have expanded (for the printout)
 
    while open_list:  # Keep searching as long as there is something left to explore
 
        # Pick the node in the open list with the lowest f_cost.
        # If two are tied, pick the one with the lower h_cost (closer to the goal).
        current = min(open_list, key=lambda n: (n.f_cost, n.h_cost))
        open_list.remove(current)  # Take it out of the open list since we are exploring it now
 
        if current.state in closed_list:  # If this point was already explored...
            continue                      # ...skip it (this is an old, more expensive copy)
 
        step += 1  # One more node expanded
        print(f"Step {step}: expanding {current}")  # Show which node we are working on
 
        if current.state == goal:  # Did we reach the goal?
            return reconstruct_path(current), current.g_cost, step  # Return path, total cost, steps
 
        closed_list.add(current.state)  # Mark this point as fully explored
 
        # Look at every neighbor we can move to from the current point
        for neighbor, move_cost in graph[current.state].items():
 
            if neighbor in closed_list:  # Skip neighbors we already explored
                continue
 
            new_g = current.g_cost + move_cost  # Cost to reach the neighbor through the current node
 
            # Only keep this route if it is the first one we found to this neighbor,
            # or if it is cheaper than the route we found before
            if neighbor not in best_g or new_g < best_g[neighbor]:
                best_g[neighbor] = new_g                               # Remember the cheaper cost
                child = Node(neighbor, current, new_g, h[neighbor])    # Make a new node for the neighbor
                open_list.append(child)                                # Add it to the open list to explore later
                print(f"    added {child}")                            # Show the new node
 
    return None, float("inf"), step  # Open list ran out and goal was never reached: no path exists

In [ ]:
if __name__ == "__main__":  # Only runs when this file is run directly
    print("A* Search")                                       # Title
    print(f"Start: {initial_state}   Goal: {goal_state}\n")  # Show start and goal
 
    path, total_cost, steps = a_star(initial_state, goal_state, state_space, heuristic)  # Run A*
 
    print("\n----- RESULT -----")
    if path:  # A path was found
        print("Path to goal :", " -> ".join(path))  # Print the path like S -> A -> B ...
        print("Total cost   :", total_cost)           # Print the total cost of the path
        print("Nodes expanded:", steps)               # Print how many nodes A* had to explore
    else:     # No path was found
        print("No path from", initial_state, "to", goal_state)